<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Amortization-of-Risk-Indicators/blob/main/Amortization_Exp1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==================================================================================================
# SECTION 5.1 — PREDICTIVE ACCURACY, POPULATION-SIZE TRANSFER,
#               AND ACCURACY AS A FUNCTION OF EXACT TRAINING-SET SIZE
#
# Neural Amortization of Exact Markovian Epidemic Risk Functionals
#
# ONE-CELL COMPLETE EXPERIMENT
#
# EXPERIMENT 5.1-A
# ----------------------------------------------------------------------------------
# Test predictive fidelity of the exact-teacher neural emulator:
#
#   (i)   truncated infection-count distribution
#             p = (P(C=0), ..., P(C=N), P(C>N))
#
#   (ii)  tail-risk curve
#             rho(c) = P(C>c)
#
#   (iii) extinction-time summaries
#             E(tau), Var(tau)
#
#   (iv)  interpolation to population sizes N that are NOT used in training,
#         but lie inside the population-size range represented in training.
#
#
# EXPERIMENT 5.1-B — LEARNING CURVE
# ----------------------------------------------------------------------------------
# Test predictive accuracy as a function of
#
#             R = number of exact Markovian training configurations.
#
# We use nested/random subsets of one fixed exact-teacher pool and keep fixed:
#
#   - neural architecture,
#   - optimizer,
#   - parameter domain,
#   - W_tau,
#   - validation set,
#   - seen-N test set,
#   - held-out-N test set.
#
# Thus R is the quantity deliberately varied.
#
#
# METRICS
# ----------------------------------------------------------------------------------
#
#   E2    = ||p_hat - p||_2
#
#   E_rho = max_c |rho_hat(c) - rho(c)|
#
#   KL    = KL(p || p_hat)
#
#   relative error in E(tau)
#
#   relative error in Var(tau)
#
# NO E1 IS USED.
#
#
# OUTPUT
# ----------------------------------------------------------------------------------
# One composite 2 x 3 publication-style figure:
#
#   A. Exact vs neural PMF at representative held-out N
#   B. Exact vs neural tail-risk curve
#   C. E2 and E_rho versus N
#   D. Extinction-time errors versus N
#   E. E2 and E_rho versus exact training-set size R
#   F. Extinction-time errors versus exact training-set size R
#
# No numerical table is generated.
#
# ==================================================================================================


# ==================================================================================================
# 0. IMPORTS
# ==================================================================================================

from __future__ import annotations

import copy
import json
import math
import pickle
import random
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np

from scipy import sparse
from scipy.sparse.linalg import splu
from scipy.stats import qmc

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt


# ==================================================================================================
# 1. EXPERIMENT CONFIGURATION
# ==================================================================================================

@dataclass
class ExperimentConfig:

    # ----------------------------------------------------------------------------------------------
    # Reproducibility
    # ----------------------------------------------------------------------------------------------
    seed: int = 20260819

    # ----------------------------------------------------------------------------------------------
    # Parameter region.
    #
    # The present manuscript leaves these ranges as TODO.
    # They are therefore explicit experimental-design choices.
    # ----------------------------------------------------------------------------------------------
    beta_range: Tuple[float, float] = (0.30, 1.50)
    gamma_range: Tuple[float, float] = (0.20, 1.00)
    omega_range: Tuple[float, float] = (0.02, 0.50)

    # i0/N
    initial_fraction_range: Tuple[float, float] = (0.02, 0.20)

    # ----------------------------------------------------------------------------------------------
    # Population sizes.
    #
    # Held-out values lie between training values.
    # Hence this is interpolation in N, not extrapolation.
    # ----------------------------------------------------------------------------------------------
    train_population_sizes: Tuple[int, ...] = (
        20, 30, 40, 50, 60
    )

    heldout_population_sizes: Tuple[int, ...] = (
        25, 35, 45, 55
    )

    # ----------------------------------------------------------------------------------------------
    # Maximum exact-teacher pool.
    #
    # The learning-curve experiment takes subsets of this pool.
    # ----------------------------------------------------------------------------------------------
    n_train_max: int = 800

    n_validation: int = 160

    n_test_seen: int = 250
    n_test_heldout: int = 250

    # ----------------------------------------------------------------------------------------------
    # Learning-curve experiment.
    #
    # R = number of exact stochastic configurations used to train the MLP.
    # ----------------------------------------------------------------------------------------------
    training_set_sizes: Tuple[int, ...] = (
        100, 200, 400, 800
    )

    # Different random subsets / network initializations.
    learning_curve_repeats: int = 3

    # ----------------------------------------------------------------------------------------------
    # Neural architecture.
    # ----------------------------------------------------------------------------------------------
    hidden_width: int = 128
    hidden_depth: int = 3

    # ----------------------------------------------------------------------------------------------
    # Optimization.
    # ----------------------------------------------------------------------------------------------
    batch_size: int = 32
    epochs: int = 500

    learning_rate: float = 1.0e-3
    weight_decay: float = 1.0e-6

    # Joint loss:
    #
    #   distribution loss + lambda_tau * extinction-time loss
    lambda_tau: float = 0.10

    early_stopping_patience: int = 50
    min_delta: float = 1.0e-6

    # ----------------------------------------------------------------------------------------------
    # Numerical constants.
    # ----------------------------------------------------------------------------------------------
    probability_tolerance: float = 1.0e-10
    kl_epsilon: float = 1.0e-12

    # ----------------------------------------------------------------------------------------------
    # Output.
    # ----------------------------------------------------------------------------------------------
    output_directory: str = "results_section_5_1"
    figure_dpi: int = 300


config = ExperimentConfig()


# Validate learning-curve design.
if max(config.training_set_sizes) > config.n_train_max:
    raise ValueError(
        "Every training_set_size must be <= n_train_max."
    )

if sorted(config.training_set_sizes) != list(config.training_set_sizes):
    raise ValueError(
        "training_set_sizes should be ordered increasingly."
    )


# ==================================================================================================
# 2. REPRODUCIBILITY
# ==================================================================================================

def set_seed(seed: int) -> None:

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    try:
        torch.use_deterministic_algorithms(
            True,
            warn_only=True
        )
    except Exception:
        pass


set_seed(config.seed)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 100)
print("SECTION 5.1")
print("=" * 100)
print(f"Device: {device}")
print(f"Exact training pool: R_max = {config.n_train_max}")
print(f"Learning-curve sizes: {config.training_set_sizes}")
print(f"Learning-curve repetitions: {config.learning_curve_repeats}")


# ==================================================================================================
# 3. OUTPUT DIRECTORY
# ==================================================================================================

output_dir = Path(
    config.output_directory
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    output_dir / "section_5_1_config.json",
    "w"
) as file:

    json.dump(
        asdict(config),
        file,
        indent=4
    )


# ==================================================================================================
# 4. TRANSIENT SIRS STATE SPACE
# ==================================================================================================

def transient_states(
    N: int
) -> Tuple[
    List[Tuple[int, int]],
    Dict[Tuple[int, int], int]
]:

    """
    Transient states

        T_N = {(s,i): i >= 1, s+i <= N}.

    Hence

        |T_N| = N(N+1)/2.
    """

    states: List[Tuple[int, int]] = []

    for i in range(1, N + 1):

        for s in range(
            0,
            N - i + 1
        ):

            states.append(
                (s, i)
            )

    state_to_index = {
        state: j
        for j, state in enumerate(states)
    }

    return states, state_to_index


# ==================================================================================================
# 5. EXACT SIRS GENERATOR AND MARKED DECOMPOSITION
# ==================================================================================================

def build_sirs_matrices(
    beta: float,
    gamma: float,
    omega: float,
    N: int
):

    """
    Construct the transient generator

        T = D0 + D1,

    where D1 contains infection transitions and D0 contains
    the remaining transient transitions plus the diagonal.

    q contains rates from transient states into the extinction set

        A = {(s,0): 0 <= s <= N}.
    """

    states, state_to_index = transient_states(
        N
    )

    M = len(states)

    T_rows = []
    T_cols = []
    T_data = []

    D1_rows = []
    D1_cols = []
    D1_data = []

    q = np.zeros(
        M,
        dtype=np.float64
    )

    for row, (s, i) in enumerate(states):

        r = N - s - i

        total_rate = 0.0

        # ------------------------------------------------------------------------------------------
        # Infection
        #
        #   (s,i) -> (s-1,i+1)
        #
        # with intensity
        #
        #   beta * s * i / N.
        # ------------------------------------------------------------------------------------------

        if s > 0:

            infection_rate = (
                beta
                * s
                * i
                / N
            )

            destination = (
                s - 1,
                i + 1
            )

            col = state_to_index[
                destination
            ]

            T_rows.append(row)
            T_cols.append(col)
            T_data.append(
                infection_rate
            )

            D1_rows.append(row)
            D1_cols.append(col)
            D1_data.append(
                infection_rate
            )

            total_rate += infection_rate

        # ------------------------------------------------------------------------------------------
        # Recovery
        #
        #   (s,i) -> (s,i-1)
        #
        # with intensity
        #
        #   gamma * i.
        #
        # When i=1, this transition reaches extinction.
        # ------------------------------------------------------------------------------------------

        recovery_rate = (
            gamma * i
        )

        if i == 1:

            q[row] += recovery_rate

        else:

            destination = (
                s,
                i - 1
            )

            col = state_to_index[
                destination
            ]

            T_rows.append(row)
            T_cols.append(col)
            T_data.append(
                recovery_rate
            )

        total_rate += recovery_rate

        # ------------------------------------------------------------------------------------------
        # Immunity loss
        #
        #   (s,i) -> (s+1,i)
        #
        # with intensity
        #
        #   omega * r.
        # ------------------------------------------------------------------------------------------

        if r > 0:

            immunity_loss_rate = (
                omega * r
            )

            destination = (
                s + 1,
                i
            )

            col = state_to_index[
                destination
            ]

            T_rows.append(row)
            T_cols.append(col)
            T_data.append(
                immunity_loss_rate
            )

            total_rate += immunity_loss_rate

        # ------------------------------------------------------------------------------------------
        # Diagonal generator entry.
        # ------------------------------------------------------------------------------------------

        T_rows.append(row)
        T_cols.append(row)
        T_data.append(
            -total_rate
        )

    T = sparse.coo_matrix(
        (
            T_data,
            (
                T_rows,
                T_cols
            )
        ),
        shape=(M, M),
        dtype=np.float64
    ).tocsc()

    D1 = sparse.coo_matrix(
        (
            D1_data,
            (
                D1_rows,
                D1_cols
            )
        ),
        shape=(M, M),
        dtype=np.float64
    ).tocsc()

    D0 = (
        T - D1
    ).tocsc()

    return (
        T,
        D0,
        D1,
        q,
        states,
        state_to_index
    )


# ==================================================================================================
# 6. EXACT MARKOVIAN TEACHER
# ==================================================================================================

def exact_markovian_targets(
    beta: float,
    gamma: float,
    omega: float,
    N: int,
    i0: int,
    probability_tolerance: float = 1.0e-10
):

    """
    Deterministic exact teacher.

    We specialize the general initial distribution in the theory
    to the deterministic initial state

        S(0) = N - i0,
        I(0) = i0,
        R(0) = 0.

    Returns
    -------
    p
        (P(C=0), ..., P(C=N), P(C>N))

    mean_tau
        E(tau)

    var_tau
        Var(tau)
    """

    if not 1 <= i0 <= N:

        raise ValueError(
            "i0 must satisfy 1 <= i0 <= N."
        )

    (
        T,
        D0,
        D1,
        q,
        states,
        state_to_index
    ) = build_sirs_matrices(
        beta=beta,
        gamma=gamma,
        omega=omega,
        N=N
    )

    M = T.shape[0]

    initial_state = (
        N - i0,
        i0
    )

    initial_index = state_to_index[
        initial_state
    ]

    alpha = np.zeros(
        M,
        dtype=np.float64
    )

    alpha[
        initial_index
    ] = 1.0

    # ==============================================================================================
    # Infection-count distribution
    #
    #       K = (-D0)^(-1) D1
    #       b = (-D0)^(-1) q
    #
    #       P(C=k) = alpha^T K^k b
    #
    #       P(C>N) = alpha^T K^(N+1) 1.
    # ==============================================================================================

    minus_D0 = (
        -D0
    ).tocsc()

    lu_D0 = splu(
        minus_D0
    )

    lu_D0_transpose = splu(
        minus_D0.T.tocsc()
    )

    b = lu_D0.solve(
        q
    )

    # v_k = (K^T)^k alpha
    v = alpha.copy()

    p = np.zeros(
        N + 2,
        dtype=np.float64
    )

    for k in range(
        N + 1
    ):

        p[k] = float(
            np.dot(
                v,
                b
            )
        )

        # ------------------------------------------------------------------------------------------
        # v_{k+1}
        #
        #       = K^T v_k
        #       = D1^T (-D0)^(-T) v_k.
        # ------------------------------------------------------------------------------------------

        temp = lu_D0_transpose.solve(
            v
        )

        v = np.asarray(
            D1.T.dot(
                temp
            )
        ).reshape(-1)

    # At this point
    #
    #       v = (K^T)^(N+1) alpha.
    #
    # Therefore
    #
    #       P(C>N)
    #       = alpha^T K^(N+1) 1
    #       = 1^T v.
    p[
        N + 1
    ] = float(
        np.sum(v)
    )

    # ----------------------------------------------------------------------------------------------
    # Numerical probability checks.
    # ----------------------------------------------------------------------------------------------

    p[
        np.abs(p)
        < probability_tolerance
    ] = 0.0

    if np.any(
        p < -probability_tolerance
    ):

        raise RuntimeError(
            "Exact solver generated a materially negative probability."
        )

    p = np.maximum(
        p,
        0.0
    )

    total_mass = float(
        p.sum()
    )

    if (
        not np.isfinite(total_mass)
        or total_mass <= 0
    ):

        raise RuntimeError(
            "Invalid exact probability vector."
        )

    if abs(
        total_mass - 1.0
    ) > 1.0e-5:

        raise RuntimeError(
            f"Probability mass sums to {total_mass:.10f}, not one."
        )

    # Correct only floating-point drift.
    p /= total_mass

    # ==============================================================================================
    # Extinction-time moments.
    #
    #       E(tau)
    #       = alpha^T (-T)^(-1) 1
    #
    #       E(tau^2)
    #       = 2 alpha^T (-T)^(-2) 1.
    # ==============================================================================================

    minus_T = (
        -T
    ).tocsc()

    lu_T = splu(
        minus_T
    )

    one = np.ones(
        M,
        dtype=np.float64
    )

    m1_vector = lu_T.solve(
        one
    )

    m2_vector = (
        2.0
        * lu_T.solve(
            m1_vector
        )
    )

    mean_tau = float(
        m1_vector[
            initial_index
        ]
    )

    second_moment_tau = float(
        m2_vector[
            initial_index
        ]
    )

    var_tau = (
        second_moment_tau
        - mean_tau ** 2
    )

    var_tau = max(
        float(var_tau),
        0.0
    )

    return (
        p,
        mean_tau,
        var_tau
    )


# ==================================================================================================
# 7. RECORD FOR ONE EXACT CONFIGURATION
# ==================================================================================================

@dataclass
class ExactRecord:

    beta: float
    gamma: float
    omega: float

    N: int
    i0: int

    p: np.ndarray

    mean_tau: float
    var_tau: float


# ==================================================================================================
# 8. SPACE-FILLING DESIGN
# ==================================================================================================

def scale_to_range(
    u: np.ndarray,
    bounds: Tuple[float, float]
):

    lower, upper = bounds

    return (
        lower
        + (upper - lower) * u
    )


def generate_configuration_design(
    n: int,
    population_sizes: Sequence[int],
    config: ExperimentConfig,
    seed: int
):

    """
    Latin-hypercube design over

        beta,
        gamma,
        omega,
        i0/N,

    with approximately balanced population-size allocation.
    """

    sampler = qmc.LatinHypercube(
        d=4,
        seed=seed
    )

    U = sampler.random(
        n=n
    )

    beta_values = scale_to_range(
        U[:, 0],
        config.beta_range
    )

    gamma_values = scale_to_range(
        U[:, 1],
        config.gamma_range
    )

    omega_values = scale_to_range(
        U[:, 2],
        config.omega_range
    )

    initial_fraction_values = scale_to_range(
        U[:, 3],
        config.initial_fraction_range
    )

    population_sizes = np.asarray(
        population_sizes,
        dtype=int
    )

    repetitions = math.ceil(
        n / len(population_sizes)
    )

    N_values = np.tile(
        population_sizes,
        repetitions
    )[:n]

    rng = np.random.default_rng(
        seed + 991
    )

    rng.shuffle(
        N_values
    )

    configurations = []

    for j in range(n):

        N = int(
            N_values[j]
        )

        i0 = int(
            np.round(
                initial_fraction_values[j]
                * N
            )
        )

        i0 = int(
            np.clip(
                i0,
                1,
                N
            )
        )

        configurations.append(
            (
                float(beta_values[j]),
                float(gamma_values[j]),
                float(omega_values[j]),
                N,
                i0
            )
        )

    return configurations


# ==================================================================================================
# 9. CONSTRUCT EXACT DATASET
# ==================================================================================================

def construct_exact_dataset(
    configurations,
    config: ExperimentConfig,
    dataset_name: str
):

    records: List[ExactRecord] = []

    start_time = time.perf_counter()

    for j, configuration in enumerate(
        configurations
    ):

        (
            beta,
            gamma,
            omega,
            N,
            i0
        ) = configuration

        (
            p,
            mean_tau,
            var_tau
        ) = exact_markovian_targets(
            beta=beta,
            gamma=gamma,
            omega=omega,
            N=N,
            i0=i0,
            probability_tolerance=
                config.probability_tolerance
        )

        records.append(
            ExactRecord(
                beta=beta,
                gamma=gamma,
                omega=omega,
                N=N,
                i0=i0,
                p=p,
                mean_tau=mean_tau,
                var_tau=var_tau
            )
        )

        if (
            (j + 1) % 25 == 0
            or
            (j + 1) == len(configurations)
        ):

            elapsed = (
                time.perf_counter()
                - start_time
            )

            print(
                f"[{dataset_name:15s}] "
                f"{j+1:4d}/{len(configurations):4d} exact configurations | "
                f"{elapsed:.1f} sec"
            )

    return records


def load_or_construct_dataset(
    cache_path: Path,
    configurations,
    config: ExperimentConfig,
    dataset_name: str
):

    if cache_path.exists():

        print(
            f"Loading cached exact dataset: {cache_path}"
        )

        with open(
            cache_path,
            "rb"
        ) as file:

            return pickle.load(
                file
            )

    records = construct_exact_dataset(
        configurations=configurations,
        config=config,
        dataset_name=dataset_name
    )

    with open(
        cache_path,
        "wb"
    ) as file:

        pickle.dump(
            records,
            file
        )

    return records


# ==================================================================================================
# 10. GENERATE FIXED TRAIN / VALIDATION / TEST DESIGNS
# ==================================================================================================

train_configurations = generate_configuration_design(
    n=config.n_train_max,
    population_sizes=
        config.train_population_sizes,
    config=config,
    seed=config.seed + 1
)

validation_configurations = generate_configuration_design(
    n=config.n_validation,
    population_sizes=
        config.train_population_sizes,
    config=config,
    seed=config.seed + 2
)

test_seen_configurations = generate_configuration_design(
    n=config.n_test_seen,
    population_sizes=
        config.train_population_sizes,
    config=config,
    seed=config.seed + 3
)

test_heldout_configurations = generate_configuration_design(
    n=config.n_test_heldout,
    population_sizes=
        config.heldout_population_sizes,
    config=config,
    seed=config.seed + 4
)


# ==================================================================================================
# 11. COMPUTE / LOAD EXACT MARKOVIAN LABELS
# ==================================================================================================

train_records_full = load_or_construct_dataset(
    cache_path=
        output_dir / "train_exact_Rmax.pkl",
    configurations=
        train_configurations,
    config=config,
    dataset_name="train_Rmax"
)

validation_records = load_or_construct_dataset(
    cache_path=
        output_dir / "validation_exact.pkl",
    configurations=
        validation_configurations,
    config=config,
    dataset_name="validation"
)

test_seen_records = load_or_construct_dataset(
    cache_path=
        output_dir / "test_seen_exact.pkl",
    configurations=
        test_seen_configurations,
    config=config,
    dataset_name="test_seen"
)

test_heldout_records = load_or_construct_dataset(
    cache_path=
        output_dir / "test_heldout_exact.pkl",
    configurations=
        test_heldout_configurations,
    config=config,
    dataset_name="test_heldout"
)


# ==================================================================================================
# 12. GLOBAL N_max
# ==================================================================================================

N_max = max(
    max(
        config.train_population_sizes
    ),
    max(
        config.heldout_population_sizes
    )
)


# ==================================================================================================
# 13. MLP BUILDER
# ==================================================================================================

def build_mlp(
    input_dimension: int,
    output_dimension: int,
    hidden_width: int,
    hidden_depth: int
):

    layers: List[nn.Module] = []

    current_dimension = (
        input_dimension
    )

    for _ in range(
        hidden_depth
    ):

        layers.append(
            nn.Linear(
                current_dimension,
                hidden_width
            )
        )

        layers.append(
            nn.SiLU()
        )

        current_dimension = (
            hidden_width
        )

    layers.append(
        nn.Linear(
            current_dimension,
            output_dimension
        )
    )

    return nn.Sequential(
        *layers
    )


# ==================================================================================================
# 14. DISCRETE-HAZARD NETWORK
# ==================================================================================================

class HazardNetwork(nn.Module):

    """
    Scalar map

        (xi,c) -> h_xi(c),

    with input

        beta,
        gamma,
        omega,
        N/N_max,
        i0/N,
        c/N.
    """

    def __init__(
        self,
        hidden_width: int,
        hidden_depth: int
    ):

        super().__init__()

        self.network = build_mlp(
            input_dimension=6,
            output_dimension=1,
            hidden_width=hidden_width,
            hidden_depth=hidden_depth
        )

    def forward(
        self,
        x: torch.Tensor
    ) -> torch.Tensor:

        logits = (
            self.network(x)
            .squeeze(-1)
        )

        return torch.sigmoid(
            logits
        )


# ==================================================================================================
# 15. EXTINCTION-TIME NETWORK
# ==================================================================================================

class ExtinctionNetwork(nn.Module):

    """
    Predicts

        E(tau), Var(tau).

    Softplus ensures positive outputs.
    """

    def __init__(
        self,
        hidden_width: int,
        hidden_depth: int,
        output_scale: np.ndarray
    ):

        super().__init__()

        self.network = build_mlp(
            input_dimension=5,
            output_dimension=2,
            hidden_width=hidden_width,
            hidden_depth=hidden_depth
        )

        self.register_buffer(
            "output_scale",
            torch.tensor(
                output_scale,
                dtype=torch.float32
            )
        )

    def forward(
        self,
        x: torch.Tensor
    ) -> torch.Tensor:

        raw = self.network(
            x
        )

        positive = (
            F.softplus(raw)
            + 1.0e-8
        )

        return (
            positive
            * self.output_scale
        )


# ==================================================================================================
# 16. INPUT FEATURES
# ==================================================================================================

def configuration_features(
    record: ExactRecord
):

    """
    Input for extinction-time head.
    """

    return torch.tensor(
        [
            record.beta,
            record.gamma,
            record.omega,
            record.N / N_max,
            record.i0 / record.N
        ],
        dtype=torch.float32,
        device=device
    )


def hazard_features(
    record: ExactRecord
):

    """
    Construct inputs for

        c = 0,...,N.
    """

    counts = torch.arange(
        0,
        record.N + 1,
        dtype=torch.float32,
        device=device
    )

    number_of_rows = (
        record.N + 1
    )

    beta = torch.full(
        (
            number_of_rows,
            1
        ),
        record.beta,
        dtype=torch.float32,
        device=device
    )

    gamma = torch.full(
        (
            number_of_rows,
            1
        ),
        record.gamma,
        dtype=torch.float32,
        device=device
    )

    omega = torch.full(
        (
            number_of_rows,
            1
        ),
        record.omega,
        dtype=torch.float32,
        device=device
    )

    normalized_N = torch.full(
        (
            number_of_rows,
            1
        ),
        record.N / N_max,
        dtype=torch.float32,
        device=device
    )

    initial_fraction = torch.full(
        (
            number_of_rows,
            1
        ),
        record.i0 / record.N,
        dtype=torch.float32,
        device=device
    )

    normalized_count = (
        counts
        / record.N
    ).unsqueeze(1)

    return torch.cat(
        [
            beta,
            gamma,
            omega,
            normalized_N,
            initial_fraction,
            normalized_count
        ],
        dim=1
    )


# ==================================================================================================
# 17. RECONSTRUCT VALID PROBABILITY DISTRIBUTION
# ==================================================================================================

def reconstruct_distribution(
    hazards: torch.Tensor
):

    """
    Given

        h(0), ..., h(N),

    reconstruct

        p(0), ..., p(N), p(>N).

    By construction:

        p_j >= 0,
        sum_j p_j = 1,

    and the implied tail probabilities are nonincreasing.
    """

    one = torch.ones(
        1,
        dtype=hazards.dtype,
        device=hazards.device
    )

    if hazards.numel() > 1:

        cumulative_survival = torch.cumprod(
            1.0
            - hazards[:-1],
            dim=0
        )

        survival_before = torch.cat(
            [
                one,
                cumulative_survival
            ],
            dim=0
        )

    else:

        survival_before = one

    probability_mass = (
        survival_before
        * hazards
    )

    overflow_probability = torch.prod(
        1.0
        - hazards
    )

    return torch.cat(
        [
            probability_mass,
            overflow_probability.reshape(1)
        ],
        dim=0
    )


def predict_distribution_tensor(
    hazard_model: HazardNetwork,
    record: ExactRecord
):

    features = hazard_features(
        record
    )

    hazards = hazard_model(
        features
    )

    return reconstruct_distribution(
        hazards
    )


# ==================================================================================================
# 18. FIXED W_tau AND OUTPUT SCALING
#
# IMPORTANT FOR THE TRAINING-SIZE EXPERIMENT:
#
# We estimate these quantities ONCE from the full exact training pool and keep them fixed
# for every value of R. Therefore changing R does not silently change the objective scaling.
# ==================================================================================================

def compute_tau_scaling(
    records: Sequence[ExactRecord]
):

    targets = np.asarray(
        [
            [
                record.mean_tau,
                record.var_tau
            ]
            for record in records
        ],
        dtype=np.float64
    )

    target_std = targets.std(
        axis=0
    )

    tau_weights = (
        1.0
        /
        (
            target_std ** 2
            + 1.0e-10
        )
    )

    output_scale = np.median(
        targets,
        axis=0
    )

    output_scale = np.maximum(
        output_scale,
        1.0e-6
    )

    return (
        tau_weights,
        output_scale
    )


tau_weights_np, tau_output_scale = compute_tau_scaling(
    train_records_full
)

tau_weights_tensor = torch.tensor(
    tau_weights_np,
    dtype=torch.float32,
    device=device
)


# ==================================================================================================
# 19. GENERIC JOINT LOSS
# ==================================================================================================

def compute_batch_loss(
    records: Sequence[ExactRecord],
    indices: np.ndarray,
    hazard_model: HazardNetwork,
    extinction_model: ExtinctionNetwork
):

    distribution_losses = []
    extinction_losses = []

    for index in indices:

        record = records[
            int(index)
        ]

        # ==========================================================================================
        # Distribution loss
        #
        #       ||p_hat - p||_2^2.
        # ==========================================================================================

        predicted_p = predict_distribution_tensor(
            hazard_model=hazard_model,
            record=record
        )

        exact_p = torch.tensor(
            record.p,
            dtype=torch.float32,
            device=device
        )

        distribution_loss = torch.sum(
            (
                predicted_p
                - exact_p
            ) ** 2
        )

        distribution_losses.append(
            distribution_loss
        )

        # ==========================================================================================
        # Extinction-time loss.
        # ==========================================================================================

        x_tau = configuration_features(
            record
        ).unsqueeze(0)

        predicted_tau = extinction_model(
            x_tau
        ).squeeze(0)

        exact_tau = torch.tensor(
            [
                record.mean_tau,
                record.var_tau
            ],
            dtype=torch.float32,
            device=device
        )

        extinction_loss = torch.sum(
            tau_weights_tensor
            *
            (
                predicted_tau
                - exact_tau
            ) ** 2
        )

        extinction_losses.append(
            extinction_loss
        )

    mean_distribution_loss = torch.stack(
        distribution_losses
    ).mean()

    mean_extinction_loss = torch.stack(
        extinction_losses
    ).mean()

    joint_loss = (
        mean_distribution_loss
        +
        config.lambda_tau
        * mean_extinction_loss
    )

    return (
        joint_loss,
        mean_distribution_loss,
        mean_extinction_loss
    )


# ==================================================================================================
# 20. GENERIC VALIDATION LOSS
# ==================================================================================================

@torch.no_grad()
def compute_validation_loss(
    validation_records: Sequence[ExactRecord],
    hazard_model: HazardNetwork,
    extinction_model: ExtinctionNetwork
):

    hazard_model.eval()
    extinction_model.eval()

    all_indices = np.arange(
        len(validation_records)
    )

    joint_loss, _, _ = compute_batch_loss(
        records=validation_records,
        indices=all_indices,
        hazard_model=hazard_model,
        extinction_model=extinction_model
    )

    return float(
        joint_loss.item()
    )


# ==================================================================================================
# 21. GENERIC TRAINING FUNCTION
# ==================================================================================================

def train_emulator(
    train_records: Sequence[ExactRecord],
    validation_records: Sequence[ExactRecord],
    training_seed: int,
    verbose: bool = True
):

    """
    Train one exact-teacher emulator.

    The same architecture, objective and validation set are used
    irrespective of R.
    """

    set_seed(
        training_seed
    )

    hazard_model = HazardNetwork(
        hidden_width=
            config.hidden_width,
        hidden_depth=
            config.hidden_depth
    ).to(device)

    extinction_model = ExtinctionNetwork(
        hidden_width=
            config.hidden_width,
        hidden_depth=
            config.hidden_depth,
        output_scale=
            tau_output_scale
    ).to(device)

    all_parameters = (
        list(
            hazard_model.parameters()
        )
        +
        list(
            extinction_model.parameters()
        )
    )

    optimizer = torch.optim.AdamW(
        all_parameters,
        lr=config.learning_rate,
        weight_decay=config.weight_decay
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=15
        )
    )

    rng = np.random.default_rng(
        training_seed + 7001
    )

    best_validation_loss = np.inf

    best_hazard_state = None
    best_extinction_state = None

    epochs_without_improvement = 0

    training_history = {
        "joint": [],
        "distribution": [],
        "extinction": [],
        "validation": []
    }

    start_time = time.perf_counter()

    for epoch in range(
        1,
        config.epochs + 1
    ):

        hazard_model.train()
        extinction_model.train()

        shuffled_indices = rng.permutation(
            len(train_records)
        )

        epoch_joint = []
        epoch_distribution = []
        epoch_extinction = []

        for start in range(
            0,
            len(shuffled_indices),
            config.batch_size
        ):

            batch_indices = shuffled_indices[
                start:
                start + config.batch_size
            ]

            optimizer.zero_grad()

            (
                joint_loss,
                distribution_loss,
                extinction_loss
            ) = compute_batch_loss(
                records=train_records,
                indices=batch_indices,
                hazard_model=hazard_model,
                extinction_model=extinction_model
            )

            joint_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                all_parameters,
                max_norm=5.0
            )

            optimizer.step()

            epoch_joint.append(
                joint_loss.item()
            )

            epoch_distribution.append(
                distribution_loss.item()
            )

            epoch_extinction.append(
                extinction_loss.item()
            )

        mean_joint = float(
            np.mean(
                epoch_joint
            )
        )

        mean_distribution = float(
            np.mean(
                epoch_distribution
            )
        )

        mean_extinction = float(
            np.mean(
                epoch_extinction
            )
        )

        validation_loss = compute_validation_loss(
            validation_records=
                validation_records,
            hazard_model=
                hazard_model,
            extinction_model=
                extinction_model
        )

        scheduler.step(
            validation_loss
        )

        training_history[
            "joint"
        ].append(
            mean_joint
        )

        training_history[
            "distribution"
        ].append(
            mean_distribution
        )

        training_history[
            "extinction"
        ].append(
            mean_extinction
        )

        training_history[
            "validation"
        ].append(
            validation_loss
        )

        improvement = (
            best_validation_loss
            - validation_loss
        )

        if improvement > config.min_delta:

            best_validation_loss = (
                validation_loss
            )

            best_hazard_state = copy.deepcopy(
                hazard_model.state_dict()
            )

            best_extinction_state = copy.deepcopy(
                extinction_model.state_dict()
            )

            epochs_without_improvement = 0

        else:

            epochs_without_improvement += 1

        if (
            verbose
            and
            (
                epoch == 1
                or
                epoch % 20 == 0
            )
        ):

            current_lr = optimizer.param_groups[
                0
            ]["lr"]

            print(
                f"R={len(train_records):4d} | "
                f"epoch={epoch:4d} | "
                f"joint={mean_joint:.4e} | "
                f"dist={mean_distribution:.4e} | "
                f"tau={mean_extinction:.4e} | "
                f"val={validation_loss:.4e} | "
                f"lr={current_lr:.2e}"
            )

        if (
            epochs_without_improvement
            >=
            config.early_stopping_patience
        ):

            if verbose:

                print(
                    f"Early stopping: "
                    f"R={len(train_records)}, epoch={epoch}"
                )

            break

    if best_hazard_state is None:

        raise RuntimeError(
            "Training did not produce a valid model."
        )

    hazard_model.load_state_dict(
        best_hazard_state
    )

    extinction_model.load_state_dict(
        best_extinction_state
    )

    training_time = (
        time.perf_counter()
        - start_time
    )

    return {
        "hazard_model":
            hazard_model,

        "extinction_model":
            extinction_model,

        "history":
            training_history,

        "training_time":
            training_time,

        "best_validation_loss":
            best_validation_loss
    }


# ==================================================================================================
# 22. TAIL PROBABILITIES
# ==================================================================================================

def tail_probabilities(
    p: np.ndarray
):

    """
    Input:

        p =
        (
            P(C=0),
            ...,
            P(C=N),
            P(C>N)
        ).

    Output:

        (
            P(C>0),
            ...,
            P(C>N)
        ).
    """

    return np.flip(
        np.cumsum(
            np.flip(
                p[1:]
            )
        )
    )


# ==================================================================================================
# 23. GENERIC NEURAL PREDICTION
# ==================================================================================================

@torch.no_grad()
def neural_prediction(
    record: ExactRecord,
    hazard_model: HazardNetwork,
    extinction_model: ExtinctionNetwork
):

    hazard_model.eval()
    extinction_model.eval()

    predicted_p = (
        predict_distribution_tensor(
            hazard_model=
                hazard_model,
            record=
                record
        )
        .detach()
        .cpu()
        .numpy()
    )

    x_tau = configuration_features(
        record
    ).unsqueeze(0)

    predicted_tau = (
        extinction_model(
            x_tau
        )
        .squeeze(0)
        .detach()
        .cpu()
        .numpy()
    )

    return (
        predicted_p,
        float(predicted_tau[0]),
        float(predicted_tau[1])
    )


# ==================================================================================================
# 24. EVALUATE ONE MODEL
#
# NO E1.
# ==================================================================================================

def evaluate_records(
    records: Sequence[ExactRecord],
    hazard_model: HazardNetwork,
    extinction_model: ExtinctionNetwork,
    split_name: str
):

    results = []

    for j, record in enumerate(
        records
    ):

        (
            predicted_p,
            predicted_mean_tau,
            predicted_var_tau
        ) = neural_prediction(
            record=
                record,
            hazard_model=
                hazard_model,
            extinction_model=
                extinction_model
        )

        exact_p = record.p

        # ==========================================================================================
        # E2
        #
        #       ||p_hat - p||_2.
        # ==========================================================================================

        E2 = float(
            np.linalg.norm(
                predicted_p
                - exact_p,
                ord=2
            )
        )

        # ==========================================================================================
        # Tail-risk error.
        # ==========================================================================================

        exact_tail = tail_probabilities(
            exact_p
        )

        predicted_tail = tail_probabilities(
            predicted_p
        )

        E_rho = float(
            np.max(
                np.abs(
                    predicted_tail
                    - exact_tail
                )
            )
        )

        # ==========================================================================================
        # KL divergence.
        # ==========================================================================================

        exact_safe = np.clip(
            exact_p,
            config.kl_epsilon,
            1.0
        )

        predicted_safe = np.clip(
            predicted_p,
            config.kl_epsilon,
            1.0
        )

        KL = float(
            np.sum(
                exact_safe
                *
                np.log(
                    exact_safe
                    /
                    predicted_safe
                )
            )
        )

        # ==========================================================================================
        # Extinction-time relative errors.
        # ==========================================================================================

        mean_tau_relative_error = (
            abs(
                predicted_mean_tau
                - record.mean_tau
            )
            /
            max(
                abs(record.mean_tau),
                1.0e-12
            )
        )

        var_tau_relative_error = (
            abs(
                predicted_var_tau
                - record.var_tau
            )
            /
            max(
                abs(record.var_tau),
                1.0e-12
            )
        )

        results.append(
            {
                "index":
                    j,

                "split":
                    split_name,

                "beta":
                    record.beta,

                "gamma":
                    record.gamma,

                "omega":
                    record.omega,

                "N":
                    record.N,

                "i0":
                    record.i0,

                "E2":
                    E2,

                "E_rho":
                    E_rho,

                "KL":
                    KL,

                "mean_tau_exact":
                    record.mean_tau,

                "mean_tau_predicted":
                    predicted_mean_tau,

                "mean_tau_relative_error":
                    float(
                        mean_tau_relative_error
                    ),

                "var_tau_exact":
                    record.var_tau,

                "var_tau_predicted":
                    predicted_var_tau,

                "var_tau_relative_error":
                    float(
                        var_tau_relative_error
                    ),

                "exact_p":
                    exact_p,

                "predicted_p":
                    predicted_p,

                "exact_tail":
                    exact_tail,

                "predicted_tail":
                    predicted_tail
            }
        )

    return results


# ==================================================================================================
# 25. SUMMARY UTILITIES
# ==================================================================================================

def summarize_metric(
    results,
    metric
):

    values = np.asarray(
        [
            result[metric]
            for result in results
        ],
        dtype=float
    )

    return {
        "median":
            float(
                np.median(values)
            ),

        "q25":
            float(
                np.quantile(
                    values,
                    0.25
                )
            ),

        "q75":
            float(
                np.quantile(
                    values,
                    0.75
                )
            ),

        "p95":
            float(
                np.quantile(
                    values,
                    0.95
                )
            ),

        "maximum":
            float(
                np.max(values)
            )
    }


def print_summary(
    results,
    title
):

    print(
        "\n"
        + "=" * 100
    )

    print(
        title
    )

    print(
        "=" * 100
    )

    metrics = [
        "E2",
        "E_rho",
        "KL",
        "mean_tau_relative_error",
        "var_tau_relative_error"
    ]

    for metric in metrics:

        summary = summarize_metric(
            results,
            metric
        )

        print(
            f"{metric:28s} | "
            f"median={summary['median']:.4e} | "
            f"IQR=({summary['q25']:.4e}, {summary['q75']:.4e}) | "
            f"p95={summary['p95']:.4e} | "
            f"max={summary['maximum']:.4e}"
        )


def metric_by_population_size(
    results,
    metric
):

    population_sizes = sorted(
        {
            result["N"]
            for result in results
        }
    )

    medians = []
    q25s = []
    q75s = []

    for N in population_sizes:

        values = np.asarray(
            [
                result[metric]
                for result in results
                if result["N"] == N
            ],
            dtype=float
        )

        medians.append(
            np.median(values)
        )

        q25s.append(
            np.quantile(
                values,
                0.25
            )
        )

        q75s.append(
            np.quantile(
                values,
                0.75
            )
        )

    return (
        np.asarray(
            population_sizes
        ),
        np.asarray(
            medians
        ),
        np.asarray(
            q25s
        ),
        np.asarray(
            q75s
        )
    )


# ==================================================================================================
# 26. EXPERIMENT 5.1-A
#
# FIT THE PRIMARY EMULATOR USING ALL R_max EXACT TRAINING CONFIGURATIONS.
# ==================================================================================================

print(
    "\n"
    + "#" * 100
)

print(
    "EXPERIMENT 5.1-A: FULL EXACT-TEACHER EMULATOR"
)

print(
    "#" * 100
)

primary_fit = train_emulator(
    train_records=
        train_records_full,
    validation_records=
        validation_records,
    training_seed=
        config.seed + 10000,
    verbose=True
)

primary_hazard_model = primary_fit[
    "hazard_model"
]

primary_extinction_model = primary_fit[
    "extinction_model"
]


# ==================================================================================================
# 27. EVALUATE PRIMARY EMULATOR
# ==================================================================================================

seen_results = evaluate_records(
    records=
        test_seen_records,
    hazard_model=
        primary_hazard_model,
    extinction_model=
        primary_extinction_model,
    split_name=
        "seen_N"
)

heldout_results = evaluate_records(
    records=
        test_heldout_records,
    hazard_model=
        primary_hazard_model,
    extinction_model=
        primary_extinction_model,
    split_name=
        "heldout_N"
)

print_summary(
    seen_results,
    "TEST ACCURACY — POPULATION SIZES USED IN TRAINING"
)

print_summary(
    heldout_results,
    "TEST ACCURACY — POPULATION SIZES HELD OUT FROM TRAINING"
)


# ==================================================================================================
# 28. SAVE PRIMARY MODEL
# ==================================================================================================

torch.save(
    {
        "hazard_model":
            primary_hazard_model.state_dict(),

        "extinction_model":
            primary_extinction_model.state_dict(),

        "config":
            asdict(config),

        "tau_weights":
            tau_weights_np,

        "tau_output_scale":
            tau_output_scale,

        "training_time":
            primary_fit[
                "training_time"
            ],

        "best_validation_loss":
            primary_fit[
                "best_validation_loss"
            ]
    },
    output_dir
    / "section_5_1_primary_model.pt"
)


# ==================================================================================================
# 29. EXPERIMENT 5.1-B
#
# ACCURACY AS A FUNCTION OF THE NUMBER R OF EXACT STOCHASTIC TRAINING EXAMPLES.
#
# For each repetition:
#
#   1. randomly permute the fixed R_max exact training pool;
#   2. use the first R configurations;
#   3. fit the same architecture;
#   4. evaluate on the same seen-N and held-out-N test sets.
#
# Hence the test distributions never change with R.
# ==================================================================================================

print(
    "\n"
    + "#" * 100
)

print(
    "EXPERIMENT 5.1-B: ACCURACY VERSUS NUMBER OF EXACT TRAINING CONFIGURATIONS R"
)

print(
    "#" * 100
)

learning_curve_records = []

for repetition in range(
    config.learning_curve_repeats
):

    # ----------------------------------------------------------------------------------------------
    # A new random ordering per repetition.
    #
    # Within one repetition, the subsets are nested:
    #
    #       R=100 subset of R=200 subset of R=400 subset of R=800.
    # ----------------------------------------------------------------------------------------------

    subset_rng = np.random.default_rng(
        config.seed
        + 20000
        + repetition
    )

    permutation = subset_rng.permutation(
        config.n_train_max
    )

    for R in config.training_set_sizes:

        print(
            "\n"
            + "-" * 100
        )

        print(
            f"Learning curve | repetition={repetition + 1}/"
            f"{config.learning_curve_repeats} | R={R}"
        )

        print(
            "-" * 100
        )

        subset_indices = permutation[
            :R
        ]

        train_subset = [
            train_records_full[
                int(index)
            ]
            for index in subset_indices
        ]

        training_seed = (
            config.seed
            + 30000
            + 1000 * repetition
            + R
        )

        fitted = train_emulator(
            train_records=
                train_subset,
            validation_records=
                validation_records,
            training_seed=
                training_seed,
            verbose=False
        )

        fitted_hazard = fitted[
            "hazard_model"
        ]

        fitted_extinction = fitted[
            "extinction_model"
        ]

        # ------------------------------------------------------------------------------------------
        # Seen-N test performance.
        # ------------------------------------------------------------------------------------------

        seen_R_results = evaluate_records(
            records=
                test_seen_records,
            hazard_model=
                fitted_hazard,
            extinction_model=
                fitted_extinction,
            split_name=
                "seen_N"
        )

        # ------------------------------------------------------------------------------------------
        # Held-out-N test performance.
        # ------------------------------------------------------------------------------------------

        heldout_R_results = evaluate_records(
            records=
                test_heldout_records,
            hazard_model=
                fitted_hazard,
            extinction_model=
                fitted_extinction,
            split_name=
                "heldout_N"
        )

        # ------------------------------------------------------------------------------------------
        # Store per-fit medians across the fixed test configurations.
        #
        # These are then summarized across repetitions.
        # ------------------------------------------------------------------------------------------

        for split_name, split_results in [
            (
                "seen_N",
                seen_R_results
            ),
            (
                "heldout_N",
                heldout_R_results
            )
        ]:

            learning_curve_records.append(
                {
                    "repetition":
                        repetition,

                    "R":
                        R,

                    "split":
                        split_name,

                    "E2":
                        summarize_metric(
                            split_results,
                            "E2"
                        )["median"],

                    "E_rho":
                        summarize_metric(
                            split_results,
                            "E_rho"
                        )["median"],

                    "KL":
                        summarize_metric(
                            split_results,
                            "KL"
                        )["median"],

                    "mean_tau_relative_error":
                        summarize_metric(
                            split_results,
                            "mean_tau_relative_error"
                        )["median"],

                    "var_tau_relative_error":
                        summarize_metric(
                            split_results,
                            "var_tau_relative_error"
                        )["median"],

                    "training_time":
                        fitted[
                            "training_time"
                        ],

                    "best_validation_loss":
                        fitted[
                            "best_validation_loss"
                        ]
                }
            )

        print(
            f"R={R:4d} | "
            f"held-out median E2="
            f"{summarize_metric(heldout_R_results, 'E2')['median']:.4e} | "
            f"held-out median E_rho="
            f"{summarize_metric(heldout_R_results, 'E_rho')['median']:.4e}"
        )

        # Free the repeated fit before training the next model.
        del fitted
        del fitted_hazard
        del fitted_extinction

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ==================================================================================================
# 30. LEARNING-CURVE SUMMARY FUNCTION
# ==================================================================================================

def learning_curve_summary(
    records,
    split,
    metric
):

    R_values = np.asarray(
        config.training_set_sizes,
        dtype=int
    )

    medians = []
    q25s = []
    q75s = []

    for R in R_values:

        values = np.asarray(
            [
                record[metric]
                for record in records
                if
                record["R"] == R
                and
                record["split"] == split
            ],
            dtype=float
        )

        medians.append(
            np.median(values)
        )

        q25s.append(
            np.quantile(
                values,
                0.25
            )
        )

        q75s.append(
            np.quantile(
                values,
                0.75
            )
        )

    return (
        R_values,
        np.asarray(medians),
        np.asarray(q25s),
        np.asarray(q75s)
    )


# ==================================================================================================
# 31. SELECT REPRESENTATIVE HELD-OUT CONFIGURATION
#
# Choose a case whose E_rho is closest to the held-out median.
# Thus it is representative rather than a best-case example.
# ==================================================================================================

heldout_Erho_values = np.asarray(
    [
        result[
            "E_rho"
        ]
        for result in heldout_results
    ]
)

median_heldout_Erho = np.median(
    heldout_Erho_values
)

representative_index = int(
    np.argmin(
        np.abs(
            heldout_Erho_values
            - median_heldout_Erho
        )
    )
)

representative = heldout_results[
    representative_index
]


# ==================================================================================================
# 32. POPULATION-SIZE SUMMARIES FOR THE PRIMARY MODEL
# ==================================================================================================

(
    N_seen_E2,
    median_seen_E2,
    q25_seen_E2,
    q75_seen_E2
) = metric_by_population_size(
    seen_results,
    "E2"
)

(
    N_held_E2,
    median_held_E2,
    q25_held_E2,
    q75_held_E2
) = metric_by_population_size(
    heldout_results,
    "E2"
)

(
    N_seen_rho,
    median_seen_rho,
    q25_seen_rho,
    q75_seen_rho
) = metric_by_population_size(
    seen_results,
    "E_rho"
)

(
    N_held_rho,
    median_held_rho,
    q25_held_rho,
    q75_held_rho
) = metric_by_population_size(
    heldout_results,
    "E_rho"
)

(
    N_seen_mean_tau,
    median_seen_mean_tau,
    q25_seen_mean_tau,
    q75_seen_mean_tau
) = metric_by_population_size(
    seen_results,
    "mean_tau_relative_error"
)

(
    N_held_mean_tau,
    median_held_mean_tau,
    q25_held_mean_tau,
    q75_held_mean_tau
) = metric_by_population_size(
    heldout_results,
    "mean_tau_relative_error"
)

(
    N_seen_var_tau,
    median_seen_var_tau,
    q25_seen_var_tau,
    q75_seen_var_tau
) = metric_by_population_size(
    seen_results,
    "var_tau_relative_error"
)

(
    N_held_var_tau,
    median_held_var_tau,
    q25_held_var_tau,
    q75_held_var_tau
) = metric_by_population_size(
    heldout_results,
    "var_tau_relative_error"
)


# ==================================================================================================
# 33. LEARNING-CURVE SUMMARIES
# ==================================================================================================

# Held-out N: E2
(
    R_E2,
    median_R_E2,
    q25_R_E2,
    q75_R_E2
) = learning_curve_summary(
    learning_curve_records,
    split="heldout_N",
    metric="E2"
)

# Held-out N: E_rho
(
    R_rho,
    median_R_rho,
    q25_R_rho,
    q75_R_rho
) = learning_curve_summary(
    learning_curve_records,
    split="heldout_N",
    metric="E_rho"
)

# Held-out N: mean tau
(
    R_mean_tau,
    median_R_mean_tau,
    q25_R_mean_tau,
    q75_R_mean_tau
) = learning_curve_summary(
    learning_curve_records,
    split="heldout_N",
    metric="mean_tau_relative_error"
)

# Held-out N: variance tau
(
    R_var_tau,
    median_R_var_tau,
    q25_R_var_tau,
    q75_R_var_tau
) = learning_curve_summary(
    learning_curve_records,
    split="heldout_N",
    metric="var_tau_relative_error"
)


# ==================================================================================================
# 34. ONE COMPOSITE FIGURE FOR SECTION 5.1
# ==================================================================================================

plt.rcParams.update(
    {
        "font.size": 10.5,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.facecolor": "white",
        "axes.facecolor": "white"
    }
)

fig, axes = plt.subplots(
    2,
    3,
    figsize=(
        17.0,
        9.5
    )
)


# ==================================================================================================
# PANEL A — REPRESENTATIVE PMF
# ==================================================================================================

ax = axes[
    0,
    0
]

N_rep = representative[
    "N"
]

exact_p_rep = representative[
    "exact_p"
]

predicted_p_rep = representative[
    "predicted_p"
]

x_values = np.arange(
    N_rep + 2
)

ax.plot(
    x_values,
    exact_p_rep,
    color="black",
    linewidth=2.0,
    marker="o",
    markersize=3.0,
    label="Exact Markovian"
)

ax.plot(
    x_values,
    predicted_p_rep,
    color="#D55E00",
    linewidth=1.8,
    linestyle="--",
    marker="s",
    markersize=2.8,
    label="Neural emulator"
)

ax.axvline(
    N_rep + 0.5,
    color="0.70",
    linewidth=1.0,
    linestyle=":"
)

tick_step = max(
    1,
    N_rep // 5
)

ticks = list(
    range(
        0,
        N_rep + 1,
        tick_step
    )
)

if (
    len(ticks) == 0
    or
    ticks[-1] != N_rep
):
    ticks.append(
        N_rep
    )

ticks.append(
    N_rep + 1
)

tick_labels = [
    str(tick)
    if tick <= N_rep
    else f">{N_rep}"
    for tick in ticks
]

ax.set_xticks(
    ticks
)

ax.set_xticklabels(
    tick_labels
)

ax.set_xlabel(
    "Cumulative infection count"
)

ax.set_ylabel(
    "Probability mass"
)

ax.set_title(
    "(A) Representative held-out population size\n"
    f"$N={N_rep}$, "
    rf"$E_2={representative['E2']:.2e}$, "
    rf"$E_\rho={representative['E_rho']:.2e}$"
)

ax.legend(
    frameon=False,
    fontsize=9
)


# ==================================================================================================
# PANEL B — REPRESENTATIVE TAIL-RISK CURVE
# ==================================================================================================

ax = axes[
    0,
    1
]

thresholds = np.arange(
    N_rep + 1
)

ax.plot(
    thresholds,
    representative[
        "exact_tail"
    ],
    color="black",
    linewidth=2.2,
    label="Exact Markovian"
)

ax.plot(
    thresholds,
    representative[
        "predicted_tail"
    ],
    color="#0072B2",
    linewidth=2.0,
    linestyle="--",
    label="Neural emulator"
)

ax.set_xlabel(
    "Threshold $c$"
)

ax.set_ylabel(
    r"$P(C>c)$"
)

ax.set_ylim(
    -0.01,
    1.01
)

ax.set_title(
    "(B) Tail-risk reconstruction\n"
    "at the same held-out configuration"
)

ax.legend(
    frameon=False,
    fontsize=9
)


# ==================================================================================================
# PANEL C — E2 AND E_rho VERSUS POPULATION SIZE
# ==================================================================================================

ax = axes[
    0,
    2
]

# E2 — seen N
ax.plot(
    N_seen_E2,
    median_seen_E2,
    color="#009E73",
    marker="o",
    linewidth=1.8,
    label=r"$E_2$, seen $N$"
)

ax.fill_between(
    N_seen_E2,
    q25_seen_E2,
    q75_seen_E2,
    color="#009E73",
    alpha=0.12
)

# E2 — held-out N
ax.plot(
    N_held_E2,
    median_held_E2,
    color="#009E73",
    marker="D",
    linestyle="--",
    linewidth=1.8,
    label=r"$E_2$, held-out $N$"
)

# E_rho — seen N
ax.plot(
    N_seen_rho,
    median_seen_rho,
    color="#CC79A7",
    marker="o",
    linewidth=1.8,
    label=r"$E_\rho$, seen $N$"
)

ax.fill_between(
    N_seen_rho,
    q25_seen_rho,
    q75_seen_rho,
    color="#CC79A7",
    alpha=0.12
)

# E_rho — held-out N
ax.plot(
    N_held_rho,
    median_held_rho,
    color="#CC79A7",
    marker="D",
    linestyle="--",
    linewidth=1.8,
    label=r"$E_\rho$, held-out $N$"
)

ax.set_yscale(
    "log"
)

ax.set_xlabel(
    "Population size $N$"
)

ax.set_ylabel(
    "Median test error"
)

ax.set_title(
    "(C) Distributional and tail-risk accuracy\n"
    "across population sizes"
)

ax.legend(
    frameon=False,
    fontsize=8.5
)


# ==================================================================================================
# PANEL D — EXTINCTION-TIME ERRORS VERSUS N
# ==================================================================================================

ax = axes[
    1,
    0
]

# Mean tau — seen
ax.plot(
    N_seen_mean_tau,
    median_seen_mean_tau,
    color="#0072B2",
    marker="o",
    linewidth=1.8,
    label=r"$E(\tau)$, seen $N$"
)

ax.fill_between(
    N_seen_mean_tau,
    q25_seen_mean_tau,
    q75_seen_mean_tau,
    color="#0072B2",
    alpha=0.12
)

# Mean tau — held-out
ax.plot(
    N_held_mean_tau,
    median_held_mean_tau,
    color="#0072B2",
    marker="D",
    linestyle="--",
    linewidth=1.8,
    label=r"$E(\tau)$, held-out $N$"
)

# Var tau — seen
ax.plot(
    N_seen_var_tau,
    median_seen_var_tau,
    color="#E69F00",
    marker="o",
    linewidth=1.8,
    label=r"$\mathrm{Var}(\tau)$, seen $N$"
)

ax.fill_between(
    N_seen_var_tau,
    q25_seen_var_tau,
    q75_seen_var_tau,
    color="#E69F00",
    alpha=0.12
)

# Var tau — held-out
ax.plot(
    N_held_var_tau,
    median_held_var_tau,
    color="#E69F00",
    marker="D",
    linestyle="--",
    linewidth=1.8,
    label=r"$\mathrm{Var}(\tau)$, held-out $N$"
)

ax.set_yscale(
    "log"
)

ax.set_xlabel(
    "Population size $N$"
)

ax.set_ylabel(
    "Median relative error"
)

ax.set_title(
    "(D) Extinction-time accuracy\n"
    "across population sizes"
)

ax.legend(
    frameon=False,
    fontsize=8.5
)


# ==================================================================================================
# PANEL E — LEARNING CURVE FOR E2 AND E_rho
#
# The test set here consists only of held-out population sizes.
# Thus this panel asks how exact training-set size influences population-size transfer.
# ==================================================================================================

ax = axes[
    1,
    1
]

ax.plot(
    R_E2,
    median_R_E2,
    color="#009E73",
    marker="o",
    linewidth=2.0,
    label=r"$E_2$"
)

ax.fill_between(
    R_E2,
    q25_R_E2,
    q75_R_E2,
    color="#009E73",
    alpha=0.16
)

ax.plot(
    R_rho,
    median_R_rho,
    color="#CC79A7",
    marker="s",
    linewidth=2.0,
    label=r"$E_\rho$"
)

ax.fill_between(
    R_rho,
    q25_R_rho,
    q75_R_rho,
    color="#CC79A7",
    alpha=0.16
)

ax.set_xscale(
    "log",
    base=2
)

ax.set_yscale(
    "log"
)

ax.set_xticks(
    R_E2
)

ax.set_xticklabels(
    [
        str(int(R))
        for R in R_E2
    ]
)

ax.set_xlabel(
    "Number of exact training configurations $R$"
)

ax.set_ylabel(
    "Median held-out-$N$ test error"
)

ax.set_title(
    "(E) Predictive accuracy versus\n"
    "exact training-set size"
)

ax.legend(
    frameon=False
)


# ==================================================================================================
# PANEL F — LEARNING CURVE FOR EXTINCTION-TIME ACCURACY
# ==================================================================================================

ax = axes[
    1,
    2
]

ax.plot(
    R_mean_tau,
    median_R_mean_tau,
    color="#0072B2",
    marker="o",
    linewidth=2.0,
    label=r"$E(\tau)$"
)

ax.fill_between(
    R_mean_tau,
    q25_R_mean_tau,
    q75_R_mean_tau,
    color="#0072B2",
    alpha=0.16
)

ax.plot(
    R_var_tau,
    median_R_var_tau,
    color="#E69F00",
    marker="s",
    linewidth=2.0,
    label=r"$\mathrm{Var}(\tau)$"
)

ax.fill_between(
    R_var_tau,
    q25_R_var_tau,
    q75_R_var_tau,
    color="#E69F00",
    alpha=0.16
)

ax.set_xscale(
    "log",
    base=2
)

ax.set_yscale(
    "log"
)

ax.set_xticks(
    R_mean_tau
)

ax.set_xticklabels(
    [
        str(int(R))
        for R in R_mean_tau
    ]
)

ax.set_xlabel(
    "Number of exact training configurations $R$"
)

ax.set_ylabel(
    "Median held-out-$N$ relative error"
)

ax.set_title(
    "(F) Extinction-time accuracy versus\n"
    "exact training-set size"
)

ax.legend(
    frameon=False
)


# ==================================================================================================
# 35. SAVE THE COMPOSITE FIGURE
# ==================================================================================================

fig.suptitle(
    "Predictive Accuracy, Population-Size Transfer, and Exact-Teacher Learning Curves",
    fontsize=15,
    y=0.995
)

plt.tight_layout(
    rect=[
        0,
        0,
        1,
        0.965
    ]
)

figure_path = (
    output_dir
    /
    "figure_section_5_1_accuracy_and_training_size.png"
)

plt.savefig(
    figure_path,
    dpi=config.figure_dpi,
    bbox_inches="tight"
)

plt.show()

print(
    "\nFigure saved to:"
)

print(
    figure_path.resolve()
)


# ==================================================================================================
# 36. SAVE RAW RESULTS
#
# No manuscript table is produced.
# These files simply preserve the numerical output for later analysis.
# ==================================================================================================

with open(
    output_dir
    /
    "section_5_1_results.pkl",
    "wb"
) as file:

    pickle.dump(
        {
            "seen_results":
                seen_results,

            "heldout_results":
                heldout_results,

            "learning_curve_records":
                learning_curve_records,

            "primary_training_history":
                primary_fit[
                    "history"
                ],

            "primary_training_time":
                primary_fit[
                    "training_time"
                ],

            "representative_index":
                representative_index
        },
        file
    )


# ==================================================================================================
# 37. FINAL DIAGNOSTIC OUTPUT
# ==================================================================================================

print(
    "\n"
    + "=" * 100
)

print(
    "SECTION 5.1 EXPERIMENTS COMPLETE"
)

print(
    "=" * 100
)

print(
    "Training population sizes:",
    config.train_population_sizes
)

print(
    "Held-out population sizes:",
    config.heldout_population_sizes
)

print(
    "Exact training-set sizes tested:",
    config.training_set_sizes
)

print(
    "Learning-curve repetitions:",
    config.learning_curve_repeats
)

print(
    "Metrics used:"
)

print(
    "  E2"
)

print(
    "  E_rho"
)

print(
    "  KL(p || p_hat)"
)

print(
    "  relative error of E(tau)"
)

print(
    "  relative error of Var(tau)"
)

print(
    "E1 is NOT computed."
)

print(
    "\nInterpretation of Experiment 5.1-B:"
)

print(
    "The horizontal axis R is the number of exact Markovian "
    "configuration-to-risk examples used to train the MLP."
)

print(
    "Validation and test configurations remain fixed for every R."
)

print(
    "Within each repetition, training subsets are nested."
)

print(
    "The shaded learning-curve regions represent the interquartile range "
    "across repeated subset/initialization fits."
)

print(
    "=" * 100
)

NameError: name 'dfv' is not defined